In [2]:
# This is a forecast notebook for GPU usage for the year 2025
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm

In [3]:
# Exploratory analisys of the data
df_full = pd.read_csv(
    "../data/gpu_in_use.csv", parse_dates=["DTime"], index_col="DTime"
)

In [4]:
# Let's cut off data after December 2025
df = df_full[df_full.index < "2025-12-19"]

In [5]:
# Let's check visually the data using plotly
fig = px.line(df, y="Usage", title="GPU Usage Over Time")
fig.update_layout(xaxis_title="Date", yaxis_title="Usage")
fig.show()

In [6]:
# Let's see smoothed data using exponential smoothing
# First let's replace hourly data with daily data using mean
"""
The span parameter defines how many recent observations primarily influence the exponentially weighted average.
With span=10, pandas calculates the smoothing factor α = 2 / (span + 1) = 2/11 ≈ 0.182.
This α is then used in the recursive formula: y_t = α × x_t + (1 - α) × y_{t-1}, where each new value y_t is 18.2% of the current observation plus 81.8% of the previous smoothed value.
The result: roughly the last 10 days carry ~86% of the total weight,
with older observations contributing progressively less.
"""


df_daily = df.resample("D").mean()
df_smoothed = df_daily.ewm(span=10, adjust=False).mean()
fig = px.line(df_smoothed, y="Usage", title="Smoothed GPU Usage Over Time")
fig.update_layout(xaxis_title="Date", yaxis_title="Usage")
fig.show()
# Let's see smoothed data using moving average

In [7]:
# Test original data
from statsmodels.tsa.stattools import (
    adfuller,
)  # pyright: ignore[reportMissingTypeStubs]

result = adfuller(df_smoothed["Usage"])
print(f"Original - p-value: {result[1]:.4f}")  # Likely > 0.05 (non-stationary)

# Test after differencing
result = adfuller(df_smoothed["Usage"].diff().dropna())
print(f"Differenced - p-value: {result[1]:.4f}")  # Should be < 0.05 (stationary)

Original - p-value: 0.9667
Differenced - p-value: 0.0002


In [8]:
# Stationary check
# First difference: removes trend
df_diff = df_smoothed.diff().dropna()
fig = px.line(df_diff, y="Usage", title="Diffed GPU Usage Over Time")
fig.update_layout(xaxis_title="Date", yaxis_title="Usage")
fig.show()

In [9]:
# Step 3: Time-Based Train/Test Split (~80% train, ~20% test)
split_date = df_smoothed.index[int(len(df_smoothed) * 0.8)]
train = df_smoothed[df_smoothed.index < split_date]
test = df_smoothed[df_smoothed.index >= split_date]

print(f"Train: {train.index.min()} to {train.index.max()} ({len(train)} days)")
print(f"Test:  {test.index.min()} to {test.index.max()} ({len(test)} days)")

Train: 2025-01-13 00:00:00 to 2025-10-11 00:00:00 (272 days)
Test:  2025-10-12 00:00:00 to 2025-12-18 00:00:00 (68 days)


In [10]:
# Step 4: Fit ARIMA & Evaluate
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Fit on training data
model = ARIMA(train["Usage"], order=(1, 1, 1))
fitted = model.fit()

# Forecast test period
forecast = fitted.get_forecast(steps=len(test))
pred_mean = forecast.predicted_mean

# Evaluate
mae = mean_absolute_error(test["Usage"], pred_mean)
rmse = root_mean_squared_error(test["Usage"], pred_mean)
print(f"MAE: {mae:.2f}, RMSE: {rmse:.2f}")

MAE: 282.49, RMSE: 312.03


In [11]:
# Contextualize the metrics
import numpy as np

# MAPE - percentage error
mape = np.mean(np.abs((test["Usage"] - pred_mean) / test["Usage"])) * 100
print(f"MAPE: {mape:.1f}%")

# Naive baseline (predict last training value)
naive_pred = train["Usage"].iloc[-1]
naive_mae = np.mean(np.abs(test["Usage"] - naive_pred))
print(f"Naive MAE: {naive_mae:.2f}")

# Improvement over naive
improvement = (naive_mae - mae) / naive_mae * 100
print(f"ARIMA improves over naive by: {improvement:.1f}%")

# Error as % of mean
mean_usage = test["Usage"].mean()
print(f"\nTest set mean: {mean_usage:.0f}")
print(f"MAE as % of mean: {mae/mean_usage*100:.1f}%")

MAPE: 12.6%
Naive MAE: 289.54
ARIMA improves over naive by: 2.4%

Test set mean: 2178
MAE as % of mean: 13.0%


In [20]:
from pmdarima import auto_arima

auto_model = auto_arima(
    train["Usage"],
    seasonal=False,  # Try True with m=7 for weekly
    stepwise=False,
    trace=True,
)
print(auto_model.summary())

# Forecast
auto_pred = auto_model.predict(n_periods=len(test))
auto_mae = np.mean(np.abs(test["Usage"] - auto_pred))
print(f"Auto-ARIMA MAE: {auto_mae:.2f}")
print(f"Improvement over naive: {(naive_mae - auto_mae) / naive_mae * 100:.1f}%")

 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=2685.399, Time=0.01 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=2562.825, Time=0.03 sec
 ARIMA(0,1,2)(0,0,0)[0] intercept   : AIC=2553.020, Time=0.03 sec
 ARIMA(0,1,3)(0,0,0)[0] intercept   : AIC=2551.586, Time=0.04 sec
 ARIMA(0,1,4)(0,0,0)[0] intercept   : AIC=2552.322, Time=0.06 sec
 ARIMA(0,1,5)(0,0,0)[0] intercept   : AIC=2553.758, Time=0.09 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=2570.781, Time=0.02 sec
 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=2552.383, Time=0.03 sec
 ARIMA(1,1,2)(0,0,0)[0] intercept   : AIC=2553.708, Time=0.05 sec
 ARIMA(1,1,3)(0,0,0)[0] intercept   : AIC=2551.738, Time=0.09 sec
 ARIMA(1,1,4)(0,0,0)[0] intercept   : AIC=2551.396, Time=0.10 sec
 ARIMA(2,1,0)(0,0,0)[0] intercept   : AIC=2548.260, Time=0.03 sec
 ARIMA(2,1,1)(0,0,0)[0] intercept   : AIC=2549.398, Time=0.04 sec
 ARIMA(2,1,2)(0,0,0)[0] intercept   : AIC=2551.254, Time=0.05 sec
 ARIMA(2,1,3)(0,0,0)[0] intercept   : AIC=2532.940, Time=0.15 sec
 ARIMA(3,1

In [21]:
# Visualize predictions vs actuals
import plotly.graph_objects as go

fig = go.Figure()

# Training data
fig.add_trace(
    go.Scatter(
        x=train.index,
        y=train["Usage"],
        mode="lines",
        name="Train",
        line=dict(color="blue"),
    )
)

# Actual test data
fig.add_trace(
    go.Scatter(
        x=test.index,
        y=test["Usage"],
        mode="lines",
        name="Actual",
        line=dict(color="green"),
    )
)

# Auto-ARIMA predictions
fig.add_trace(
    go.Scatter(
        x=test.index,
        y=auto_pred,
        mode="lines",
        name="Auto-ARIMA",
        line=dict(color="red", dash="dash"),
    )
)

# Naive baseline
_ = fig.add_hline(
    y=naive_pred, line_dash="dot", line_color="gray", annotation_text="Naive"
)

fig.update_layout(
    title="Auto-ARIMA Predictions vs Actuals", xaxis_title="Date", yaxis_title="Usage"
)
fig.show()

In [22]:
# Forecast with confidence intervals using pmdarima directly
print(f"Best order from auto_arima: {auto_model.order}")

# Get forecast with confidence intervals from pmdarima
forecast_vals, conf_int = auto_model.predict(
    n_periods=len(test), return_conf_int=True, alpha=0.05
)

# Plot with confidence intervals
fig = go.Figure()

# Training data
_ = fig.add_trace(
    go.Scatter(
        x=train.index,
        y=train["Usage"],
        mode="lines",
        name="Train",
        line=dict(color="blue"),
    )
)

# Actual test data
_ = fig.add_trace(
    go.Scatter(
        x=test.index,
        y=test["Usage"],
        mode="lines",
        name="Actual",
        line=dict(color="green"),
    )
)

# Forecast
_ = fig.add_trace(
    go.Scatter(
        x=test.index,
        y=forecast_vals,
        mode="lines",
        name="Forecast",
        line=dict(color="red"),
    )
)

# Confidence interval (shaded area)
_ = fig.add_trace(
    go.Scatter(
        x=list(test.index) + list(test.index[::-1]),
        y=list(conf_int[:, 1]) + list(conf_int[:, 0][::-1]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        name="95% CI",
    )
)

fig.update_layout(
    title="Forecast with 95% Confidence Interval",
    xaxis_title="Date",
    yaxis_title="Usage",
)
fig.show()

Best order from auto_arima: (2, 1, 3)


In [23]:
# Future forecast: Train on pre-shutdown data, display full actuals + 3-month forecast
# Load full data (includes shutdown period Dec 19 - Jan 7)
df_full = pd.read_csv(
    "../data/gpu_in_use.csv", parse_dates=["DTime"], index_col="DTime"
)
df_full_daily = df_full.resample("D").mean()
df_full_smoothed = df_full_daily.ewm(span=10, adjust=False).mean()

# Train model on pre-shutdown data (df - ends Dec 19)
# df_smoothed already exists from earlier cells
model_future = auto_arima(
    df_smoothed["Usage"],
    seasonal=False,
    stepwise=False,  # This is a critical paramter - slow but optimal if False
    suppress_warnings=True,
)
print(f"Model order: {model_future.order}")

# Forecast 90 days (3 months) from Jan 13, 2026
forecast_days = 90
future_forecast, future_conf_int = model_future.predict(
    n_periods=forecast_days, return_conf_int=True, alpha=0.05
)

# Create future date index
last_date = df_full_smoothed.index.max()
future_dates = pd.date_range(
    start=last_date + pd.Timedelta(days=1), periods=forecast_days, freq="D"
)

# Plot
fig = go.Figure()

# Full historical data (includes shutdown)
fig.add_trace(
    go.Scatter(
        x=df_full_smoothed.index,
        y=df_full_smoothed["Usage"],
        mode="lines",
        name="Actual (smoothed)",
        line=dict(color="blue"),
    )
)

# Future forecast
fig.add_trace(
    go.Scatter(
        x=future_dates,
        y=future_forecast,
        mode="lines",
        name="Forecast",
        line=dict(color="red"),
    )
)

# Confidence interval
fig.add_trace(
    go.Scatter(
        x=list(future_dates) + list(future_dates[::-1]),
        y=list(future_conf_int[:, 1]) + list(future_conf_int[:, 0][::-1]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        name="95% CI",
    )
)

# Mark shutdown period
fig.add_vrect(
    x0="2025-12-19",
    x1="2026-01-07",
    fillcolor="gray",
    opacity=0.2,
    annotation_text="Shutdown",
    annotation_position="top left",
)

fig.update_layout(
    title="GPU Usage: Historical Data + 3-Month Forecast (Jan-Apr 2026)",
    xaxis_title="Date",
    yaxis_title="Usage",
)
fig.show()

Model order: (2, 1, 3)


In [25]:
# Brute-force ARIMA parameter search vs auto_arima
from scipy.optimize import brute
import warnings

warnings.filterwarnings("ignore")


def arima_objective(order, endog):
    """Fits ARIMA and returns AIC (lower is better)."""
    p, d, q = int(order[0]), int(order[1]), int(order[2])
    try:
        model = ARIMA(endog, order=(p, d, q))
        model_fit = model.fit()
        return model_fit.aic
    except Exception:
        return 1e10


# Run brute-force search on training data
# Ranges: p=0-4, d=0-2, q=0-4
ranges = (slice(0, 5, 1), slice(0, 3, 1), slice(0, 5, 1))

print("Running brute-force grid search (this may take a minute)...")
result = brute(
    func=arima_objective,
    ranges=ranges,
    args=(train["Usage"],),
    finish=None,
    full_output=True,
)

best_params = result[0]
min_aic = result[1]
brute_order = (int(best_params[0]), int(best_params[1]), int(best_params[2]))

print(f"\n=== Comparison ===")
print(f"Brute-force best order: {brute_order}, AIC: {min_aic:.2f}")
print(f"Auto-ARIMA best order:  {auto_model.order}, AIC: {auto_model.aic():.2f}")

# Test brute-force model on test set
brute_model = ARIMA(train["Usage"], order=brute_order)
brute_fitted = brute_model.fit()
brute_pred = brute_fitted.forecast(steps=len(test))
brute_mae = np.mean(np.abs(test["Usage"] - brute_pred))

print(f"\nTest set MAE comparison:")
print(f"Brute-force MAE: {brute_mae:.2f}")
print(f"Auto-ARIMA MAE:  {auto_mae:.2f}")
print(f"Naive MAE:       {naive_mae:.2f}")

Running brute-force grid search (this may take a minute)...

=== Comparison ===
Brute-force best order: (3, 2, 4), AIC: 2526.46
Auto-ARIMA best order:  (2, 1, 3), AIC: 2532.94

Test set MAE comparison:
Brute-force MAE: 151.60
Auto-ARIMA MAE:  167.06
Naive MAE:       289.54


In [26]:
# 3-Month Forecast using ARIMA(3, 2, 4) from brute-force search
best_order = (3, 2, 4)

# Fit on pre-shutdown data (df_smoothed)
model_brute = ARIMA(df_smoothed["Usage"], order=best_order)
fitted_brute = model_brute.fit()
print(f"ARIMA{best_order} fitted. AIC: {fitted_brute.aic:.2f}")

# Forecast 90 days ahead
forecast_days = 90
forecast_result = fitted_brute.get_forecast(steps=forecast_days)
forecast_mean = forecast_result.predicted_mean
conf_int_brute = forecast_result.conf_int(alpha=0.05)

# Create future date index
last_date = df_full_smoothed.index.max()
future_dates = pd.date_range(
    start=last_date + pd.Timedelta(days=1), periods=forecast_days, freq="D"
)

# Plot
fig = go.Figure()

# Full historical data (includes shutdown)
fig.add_trace(
    go.Scatter(
        x=df_full_smoothed.index,
        y=df_full_smoothed["Usage"],
        mode="lines",
        name="Actual (smoothed)",
        line=dict(color="blue"),
    )
)

# ARIMA forecast
fig.add_trace(
    go.Scatter(
        x=future_dates,
        y=forecast_mean.values,
        mode="lines",
        name=f"ARIMA{best_order} Forecast",
        line=dict(color="red"),
    )
)

# Confidence interval
fig.add_trace(
    go.Scatter(
        x=list(future_dates) + list(future_dates[::-1]),
        y=list(conf_int_brute.iloc[:, 1]) + list(conf_int_brute.iloc[:, 0][::-1]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        name="95% CI",
    )
)

# Shutdown marker
fig.add_vrect(
    x0="2025-12-19",
    x1="2026-01-07",
    fillcolor="gray",
    opacity=0.2,
    annotation_text="Shutdown",
    annotation_position="top left",
)

fig.update_layout(
    title=f"ARIMA{best_order} (Brute-Force Best): 3-Month Forecast",
    xaxis_title="Date",
    yaxis_title="Usage",
    yaxis=dict(range=[0, 4000], dtick=500),
    height=600,
)
fig.show()

ARIMA(3, 2, 4) fitted. AIC: 3333.36


In [27]:
# Cross-Validation: Compare ARIMA(3,2,4) vs ARIMA(2,1,3) using Expanding Window
from sklearn.model_selection import TimeSeriesSplit


def cv_arima(data, order, n_splits=5):
    """Run expanding window CV for a given ARIMA order."""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(data)):
        train_cv = data.iloc[train_idx]
        test_cv = data.iloc[test_idx]

        try:
            model = ARIMA(train_cv, order=order)
            fitted = model.fit()
            preds = fitted.forecast(steps=len(test_cv))
            mae = np.mean(np.abs(test_cv.values - preds.values))
            scores.append(mae)
            print(
                f"  Fold {fold+1}: Train={len(train_cv)}, Test={len(test_cv)}, MAE={mae:.2f}"
            )
        except Exception as e:
            print(f"  Fold {fold+1}: Failed - {e}")
            scores.append(np.nan)

    return scores


# Run CV for both models
print("=" * 60)
print("ARIMA(3,2,4) - Brute-Force Best")
print("=" * 60)
scores_brute = cv_arima(df_smoothed["Usage"], order=(3, 2, 4))

print("\n" + "=" * 60)
print("ARIMA(2,1,3) - Auto-ARIMA Best")
print("=" * 60)
scores_auto = cv_arima(df_smoothed["Usage"], order=(2, 1, 3))

# Summary comparison
print("\n" + "=" * 60)
print("CROSS-VALIDATION SUMMARY")
print("=" * 60)
print(
    f"ARIMA(3,2,4): Mean MAE = {np.nanmean(scores_brute):.2f} ± {np.nanstd(scores_brute):.2f}"
)
print(
    f"ARIMA(2,1,3): Mean MAE = {np.nanmean(scores_auto):.2f} ± {np.nanstd(scores_auto):.2f}"
)

# Determine winner
if np.nanmean(scores_brute) < np.nanmean(scores_auto):
    print("\n✅ ARIMA(3,2,4) wins on cross-validation!")
else:
    print("\n✅ ARIMA(2,1,3) wins on cross-validation!")

ARIMA(3,2,4) - Brute-Force Best
  Fold 1: Train=60, Test=56, MAE=343.21
  Fold 2: Train=116, Test=56, MAE=35.06
  Fold 3: Train=172, Test=56, MAE=123.69
  Fold 4: Train=228, Test=56, MAE=254.67
  Fold 5: Train=284, Test=56, MAE=235.18

ARIMA(2,1,3) - Auto-ARIMA Best
  Fold 1: Train=60, Test=56, MAE=130.23
  Fold 2: Train=116, Test=56, MAE=81.77
  Fold 3: Train=172, Test=56, MAE=102.70
  Fold 4: Train=228, Test=56, MAE=364.80
  Fold 5: Train=284, Test=56, MAE=105.92

CROSS-VALIDATION SUMMARY
ARIMA(3,2,4): Mean MAE = 198.36 ± 107.47
ARIMA(2,1,3): Mean MAE = 157.08 ± 104.99

✅ ARIMA(2,1,3) wins on cross-validation!
